In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2
import warnings

# Biogeme core
import biogeme.biogeme as bio
from biogeme import models
from biogeme.expressions import Beta, Variable, bioDraws

import biogeme.database as db

warnings.filterwarnings("ignore")

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
C:\Users\vladimir.koev\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


# Phase 4 — Multinomial Logit (MNL) Model Estimation

## 4.0 Overview

This phase estimates commuter mode choice models separately for Amsterdam and London using the Random Utility Maximisation (RUM) framework and the Multinomial Logit (MNL) specification (McFadden, 1974). The dependent variable is the chosen commute mode — car (reference), public transport, or bicycle — and estimation proceeds by maximum likelihood via Biogeme (Bierlaire, 2020).

We follow the alternative-specific coefficient approach recommended by Ben-Akiva & Lerman (1985): socioeconomic attributes (income, car ownership, driving licence, age, gender) enter the utility functions of PT and bicycle relative to car. Car mode choice  coefficients are normalised to zero for identification. Travel time remains a generic coefficient in this phase. Two mode-specific variables are added based on theoretical priors: number of transfers (PT only — each transfer adds disutility; Ortúzar & Willumsen, 2011) and trip distance (applied to bicycle only — physical effort increases with distance adding disutility).

We are going to estimate two nested specifications per city:
1. Experiment A: baseline
2. Experiment B: adding peak-hour effects.
  
The final model per city is selected via a likelihood-ratio test, McFadden's pseudo-$R^2$, and coefficient stability. Results feed directly into hypothesis testing in Phase 5.

## 4.1 Data Preparation for MNL

The first step is to create model-ready datasets for each city with identical coding rules. One row per observed choice, numerical coding of alternatives, clean covariate ranges, and no missing values in model variables. This step also enforces the same dependent variable mapping across cities, so estimated coefficients are directly comparable.

The code below constructs Amsterdam and London worksets from the harmonised cleaned datasets, applies defensible variable typing, and creates a shared alternative coding: car = 1, pt = 2, bike = 3.

In [2]:
amst_data = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\amst_processed.csv")

In [3]:
lnd_data = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\lnd_processed.csv")

In [4]:
pd.set_option('display.max_columns', None)
lnd_data.head(10)

,person_id,trip_id,trip_purpose,mode_detailed,travel_time_min,distance_km,departure_hour,weekday,is_holiday,age_band,gender,income_quintile,has_driving_license,n_cars_household,weight_trip,weight_person,city,is_peak,chosen_mode,n_transfers,has_transfer,n_legs
0,2023009631,2023109722,1,10,60.0,11.265408,0,3,4,8,1,1,0,0,0.986298,0.916839,London,0,pt,0,0,1
1,2023007220,2023083399,1,10,40.0,17.702784,23,5,3,6,2,5,0,3,1.032084,1.143973,London,0,pt,0,0,1
2,2023000996,2023011523,1,10,15.0,3.379622,0,1,3,6,1,3,0,0,1.461324,1.431693,London,0,pt,0,0,1
3,2023008133,2023092873,1,10,47.0,6.759245,-1,5,4,5,2,2,0,1,1.396613,1.308343,London,0,pt,2,1,3
4,2023014646,2023170543,1,10,60.0,24.944832,-1,3,3,5,2,5,1,4,0.691940,0.698998,London,0,pt,1,1,2
5,2023002887,2023033439,1,2,30.0,8.851392,5,4,4,5,1,4,0,1,1.678506,1.286774,London,0,bike,0,0,1
6,2023000323,2023003673,1,7,35.0,24.140160,5,1,3,8,2,5,1,2,1.073553,1.142800,London,0,pt,1,1,2
7,2023010282,2023118043,1,3,31.0,20.921472,5,2,3,8,1,2,1,2,0.537043,0.587190,London,0,car,0,0,1
8,2023011915,2023138347,1,7,45.0,5.954573,6,2,4,6,1,2,1,1,1.564170,1.254609,London,1,pt,0,0,1
9,2023014242,2023165626,1,10,20.0,3.218688,6,4,3,5,1,5,1,0,1.481708,1.272565,London,1,pt,0,0,1


In [5]:
mode_to_id = {"car": 1, "pt": 2, "bike": 3}

# MNL Variables
model_columns = [
    "chosen_mode",
    "travel_time_min",
    "distance_km",         
    "n_transfers",           
    "income_quintile",
    "has_driving_license",
    "n_cars_household",
    "age_band",
    "gender",
    "is_peak",
    "weight_trip",
]

In [6]:
def build_city_workset(city_data, city_name):
    
    ws = city_data[model_columns].copy()

    # 1. Drop rows with any missing value in model columns
    # n_before = len(ws)
    # ws = ws.dropna(subset=model_columns)
    # print(f"{city_name}: dropped {n_before - len(ws):,} rows with missing model vars")

    # 2. Encode dependent variable as integer
    ws["choice_code"] = ws.chosen_mode.map(mode_to_id).astype(int)

    # 3. Cast all numeric columns explicitly
    numeric_cols = [
        "travel_time_min", "distance_km", "n_transfers",
        "income_quintile", "has_driving_license",
        "n_cars_household", "age_band", "gender",
        "is_peak", "weight_trip", "choice_code",
    ]
    ws[numeric_cols] = ws[numeric_cols].apply(pd.to_numeric, errors = "coerce")
    ws = ws.dropna(subset = numeric_cols)

    # 4. Keep only final modelling columns
    keep = ["choice_code"] + [c for c in numeric_cols if c != "choice_code"]
    ws = ws[keep].reset_index(drop=True)

    # 5. Summary
    print(f"{city_name}: {ws.shape[0]:,} rows ready for MNL")
    return ws


# --- Build worksets ---
amsterdam_workset = build_city_workset(amst_data, "Amsterdam")
london_workset    = build_city_workset(lnd_data, "London")

Amsterdam: 681 rows ready for MNL
London: 3,315 rows ready for MNL


In [7]:
amsterdam_workset.columns == london_workset.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True])

## 4.2 Model Specification and Statistical Tests

In this step we are going to prepare for two experements per city described as below:

1. Experiment A (baseline): We includes travel time and socioeconomic variables in the utility function of each transport mode.
2. Experiment B (robustness): We are add peak-hour effects into the utility function

The utility functions in both experiments use alternative-specific constants (ASC), with car as the reference alternative (${ASC}_{car}=0$) for identification. In discrete choice models, we only care about the difference in utility between alternatives, not the total utility. (Ben-Akiva & Lerman, 1985, §5.2)

It is important to mention what ASC is. Generally, an ASC represents the average impact of all unobserved factors on the utility of a specific mode. For example,  the "Hidden" factors in our model. Our model includes travel time but it doesn't include variables like "comfort," "prestige," "privacy," or "safety".  If travel time and cost were exactly the same for all modes, would people still prefer one over the other? The ASC captures that inherent bias.

For model comparison, we are going to calculate the differences in log-likelihood and pseudo-$R^2$, and inspect coefficient sign/significance stability. For each parameter $\beta_k$, the hypothesis test is:

$$
H_0: \beta_k = 0 \quad \text{vs} \quad H_1: \beta_k \neq 0
$$

at $\alpha = 0.05$. A p-value below $\alpha$ implies rejection of $H_0$, indicating statistical evidence that the covariate is associated with mode choice.


In [9]:
def estimate_city_mnl(city_workset, city_name, include_peak=False):
    tag = "exp_b_peak" if include_peak else "exp_a_base"
    database = db.Database(f"{city_name.lower()}_{tag}", city_workset)

    # --- Biogeme variable handles ---
    choice = Variable("choice_code")
    travel_time_min = Variable("travel_time_min")
    distance_km = Variable("distance_km")
    income_quintile = Variable("income_quintile")
    has_driving_lic = Variable("has_driving_license")
    n_cars = Variable("n_cars_household")
    age_band = Variable("age_band")
    gender = Variable("gender")
    is_peak = Variable("is_peak")

    # ASC_car = 0)
    asc_pt = Beta("asc_pt",   0, None, None, 0)
    asc_bike = Beta("asc_bike", 0, None, None, 0)

    # travel-time coefficient
    b_time_pt = Beta("b_time_pt", 0, None, None, 0)
    b_time_bike = Beta("b_time_bike", 0, None, None, 0)

    # Alternative-specific socioeconomic (car = 0)
    b_income_pt = Beta("b_income_pt",   0, None, None, 0)
    b_income_bike = Beta("b_income_bike", 0, None, None, 0)

    b_license_pt = Beta("b_license_pt",   0, None, None, 0)
    b_license_bike = Beta("b_license_bike", 0, None, None, 0)

    b_cars_pt = Beta("b_cars_pt",   0, None, None, 0)
    b_cars_bike = Beta("b_cars_bike", 0, None, None, 0)

    b_age_pt = Beta("b_age_pt",   0, None, None, 0)
    b_age_bike = Beta("b_age_bike", 0, None, None, 0)

    b_gender_pt = Beta("b_gender_pt",   0, None, None, 0)
    b_gender_bike = Beta("b_gender_bike", 0, None, None, 0)

    # Mode-specific LoS variables
    b_dist_bike = Beta("b_dist_bike", 0, None, None, 0)   # bike only

    # Optional peak-hour (Experiment B) — alternative-specific ===
    if include_peak:
        b_peak_pt = Beta("b_peak_pt",   0, None, None, 0)
        b_peak_bike = Beta("b_peak_bike", 0, None, None, 0)
    else:
        b_peak_pt = 0
        b_peak_bike = 0

    # Utility functions:
    
    # V_car: reference — only generic time enters
    v_car = 0

    # V_pt: ASC + generic time + socioeconomic (alt-specific) + transfers
    v_pt = (
        asc_pt
        + b_time_pt * travel_time_min
        + b_income_pt * income_quintile
        + b_license_pt * has_driving_lic
        + b_cars_pt * n_cars
        + b_age_pt * age_band
        + b_gender_pt * gender
        + b_peak_pt * is_peak
    )

    # V_bike: ASC + generic time + socioeconomic (alt-specific) + distance
    v_bike = (
        asc_bike
        + b_time_bike * travel_time_min
        + b_income_bike * income_quintile
        + b_license_bike * has_driving_lic
        + b_cars_bike * n_cars
        + b_age_bike * age_band
        + b_gender_bike * gender
        + b_dist_bike * distance_km
        + b_peak_bike * is_peak
    )

    # --- Choice model ---
    V  = {1: v_car, 2: v_pt, 3: v_bike}
    av = {1: 1, 2: 1, 3: 1}          # all modes available
    logprob = models.loglogit(V, av, choice)

    # --- Biogeme estimation ---
    biogeme_model = bio.BIOGEME(database, logprob, generate_html=False)
    biogeme_model.modelName = f"mnl_{city_name.lower()}_{tag}"
    
    results = biogeme_model.estimate()

    # --- Extract results ---
    beta_table = results.get_estimated_parameters()
    beta_table = beta_table.reset_index().rename(columns = {"index": "parameter"})
    beta_table["city"] = city_name
    beta_table["spec"] = tag
    
    # --- Fit statistics ---
    n_obs = city_workset.shape[0]
    ll_null = -n_obs * np.log(3)

    general_stats = results.get_general_statistics()

    # Extract LL and n_params — values may be tuples like (value, precision_str)
    ll_star = None
    n_params = None
    for key, val in general_stats.items():
        key_str = str(key).lower()
        # val is often a tuple: (numeric_value, format_string)
        numeric_val = val[0] if isinstance(val, (tuple, list)) else val
        if 'final log likelihood' in key_str:
            ll_star = float(numeric_val)
        if 'number of estimated parameters' in key_str:
            n_params = int(numeric_val)

    # Fallback
    if ll_star is None:
        try:
            ll_star = float(results.data.logLike)
        except Exception:
            ll_star = np.nan
    if n_params is None:
        n_params = len(beta_table)

    rho2 = 1.0 - (ll_star / ll_null)

    fit_stats = {
        "city": city_name, "spec": tag,
        "n_obs": n_obs, "n_params": n_params,
        "LL_null": round(ll_null, 2),
        "LL_star": round(ll_star, 2),
        "rho2": round(rho2, 4),
    }

    # Print Resutlts
    print(f"\n{'='*60}")
    print(f"✓ {city_name} | {tag} | n={n_obs:,} | LL*={ll_star:.2f} | ρ²={rho2:.4f}")
    print(f"{'='*60}")
    print(beta_table.to_string(index=False))

    return biogeme_model, results, beta_table, fit_stats

## 4.3 Amsterdam — Model Estimation

We now estimate the two specifications for Amsterdam (n = 681). Given the relatively small sample, we expect wider confidence intervals and potentially insignificant coefficients for some socioeconomic variables. Amsterdam's mode split is dominated  by bycicle (60%), which differs sharply from London and should be reflected in the ASC estimates. Experiment A provides the baseline; Experiment B adds peak-hour alternative-specific effects to test whether morning/evening rush affects mode preference for Amsterdam commuters.

In [10]:
amst_model_a1, amst_results_a1, amst_beta_a1, amst_fit_a1 = estimate_city_mnl(
    amsterdam_workset, "Amsterdam", include_peak = False
)


✓ Amsterdam | exp_a_base | n=681 | LL*=-341.67 | ρ²=0.5433
 parameter           Name      Value  Robust std err.  Robust t-stat.  Robust p-value      city       spec
         0         asc_pt   7.967589         1.157180        6.885347    5.764722e-12 Amsterdam exp_a_base
         1      b_time_pt   0.053437         0.011806        4.526263    6.003595e-06 Amsterdam exp_a_base
         2    b_income_pt   0.087140         0.135224        0.644414    5.193068e-01 Amsterdam exp_a_base
         3   b_license_pt -10.729147         0.335135      -32.014370    0.000000e+00 Amsterdam exp_a_base
         4      b_cars_pt  -1.706178         0.318527       -5.356455    8.487081e-08 Amsterdam exp_a_base
         5       b_age_pt  -0.058130         0.117284       -0.495637    6.201508e-01 Amsterdam exp_a_base
         6    b_gender_pt   0.996941         0.333786        2.986762    2.819490e-03 Amsterdam exp_a_base
         7       asc_bike  12.912407         0.840390       15.364784    0.000000e+0

In [11]:
amst_model_b1, amst_results_b1, amst_beta_b1, amst_fit_b1 = estimate_city_mnl(
    amsterdam_workset, "Amsterdam", include_peak = True
)


✓ Amsterdam | exp_b_peak | n=681 | LL*=-340.88 | ρ²=0.5444
 parameter           Name      Value  Robust std err.  Robust t-stat.  Robust p-value      city       spec
         0         asc_pt   7.792436         1.159212        6.722181    1.790235e-11 Amsterdam exp_b_peak
         1      b_time_pt   0.054547         0.012314        4.429720    9.435531e-06 Amsterdam exp_b_peak
         2    b_income_pt   0.112968         0.136435        0.828002    4.076693e-01 Amsterdam exp_b_peak
         3   b_license_pt -10.326713         0.339005      -30.461788    0.000000e+00 Amsterdam exp_b_peak
         4      b_cars_pt  -1.727236         0.317073       -5.447433    5.110195e-08 Amsterdam exp_b_peak
         5       b_age_pt  -0.077329         0.118069       -0.654952    5.124984e-01 Amsterdam exp_b_peak
         6    b_gender_pt   1.015665         0.334057        3.040396    2.362669e-03 Amsterdam exp_b_peak
         7      b_peak_pt  -0.398032         0.336720       -1.182085    2.371721e-0

## 4.4 London — Model Estimation

London's sample (n=3,394) is roughly five times larger than Amsterdam's, providing substantially better statistical base. The mode split is PT-dominant (53%), with car at 39% and bicycle at only 8%. We expect stronger significance across all coefficients and tighter confidence intervals. The same two specifications are estimated, preserving strict comparability with Amsterdam.

In [12]:
lnd_model_a1, lnd_results_a1, lnd_beta_a1, lnd_fit_a1 = estimate_city_mnl(
    london_workset, "London", include_peak=False
)


✓ London | exp_a_base | n=3,315 | LL*=-1940.47 | ρ²=0.4672
 parameter           Name     Value  Robust std err.  Robust t-stat.  Robust p-value   city       spec
         0         asc_pt  5.630247         0.642413        8.764218    0.000000e+00 London exp_a_base
         1      b_time_pt  0.036808         0.003264       11.275584    0.000000e+00 London exp_a_base
         2    b_income_pt  0.338074         0.035664        9.479311    0.000000e+00 London exp_a_base
         3   b_license_pt -5.346697         0.486798      -10.983405    0.000000e+00 London exp_a_base
         4      b_cars_pt -1.035415         0.076790      -13.483771    0.000000e+00 London exp_a_base
         5       b_age_pt -0.434476         0.043122      -10.075497    0.000000e+00 London exp_a_base
         6    b_gender_pt  0.743917         0.104105        7.145841    8.943957e-13 London exp_a_base
         7       asc_bike  3.840949         0.792038        4.849451    1.238035e-06 London exp_a_base
         8   

In [13]:
lnd_model_b1, lnd_results_b1, lnd_beta_b1, lnd_fit_b1 = estimate_city_mnl(
    london_workset, "London", include_peak=True
)


✓ London | exp_b_peak | n=3,315 | LL*=-1935.06 | ρ²=0.4687
 parameter           Name     Value  Robust std err.  Robust t-stat.  Robust p-value   city       spec
         0         asc_pt  5.386380         0.652628        8.253365    2.220446e-16 London exp_b_peak
         1      b_time_pt  0.036401         0.003231       11.264946    0.000000e+00 London exp_b_peak
         2    b_income_pt  0.318476         0.036243        8.787131    0.000000e+00 London exp_b_peak
         3   b_license_pt -5.435687         0.489658      -11.100985    0.000000e+00 London exp_b_peak
         4      b_cars_pt -1.025756         0.076896      -13.339532    0.000000e+00 London exp_b_peak
         5       b_age_pt -0.418532         0.043687       -9.580303    0.000000e+00 London exp_b_peak
         6    b_gender_pt  0.732602         0.104205        7.030420    2.059242e-12 London exp_b_peak
         7      b_peak_pt  0.404529         0.124797        3.241508    1.188992e-03 London exp_b_peak
         8   

### Model Variables Adjustments

From the intial runs of the MNL models we see a significantly high fit of the MNL choice model with McFadden's pseudo-$R^2$ > 0.5 for Amsterdam runs and >0.45 for London runs. Even though, these values can be considered as very good and accurate model, usually the$R^2$ are much lower:

While a score of 0.54 indicates that our model is very good at predicting choices, it can actually flags issues with our modelling approach that could make our research invalid. A potential option is that we might have included a variable that is actually just a different version of the choice itself. This might be the case with the variables __b_licence__ and __b_cars__  which can almost perfectly predicts a choice (e.g., everyone who owns a car always commute by car). We can suggest so from the variable values being the only ones above 1.0. The b_licence is even higher than 10.0 for Amsterdam experiments. 

We will exclude those two variables from our modelling set because they might "capture" the whole choice.      

In [10]:
def estimate_city_mnl_v2(city_workset, city_name, include_peak=False):
    tag = "exp_b_peak" if include_peak else "exp_a_base"
    database = db.Database(f"{city_name.lower()}_{tag}", city_workset)

    # --- Biogeme variable handles ---
    choice = Variable("choice_code")
    travel_time_min = Variable("travel_time_min")
    distance_km = Variable("distance_km")
    income_quintile = Variable("income_quintile")
    # has_driving_lic = Variable("has_driving_license")
    # n_cars = Variable("n_cars_household")
    age_band = Variable("age_band")
    gender = Variable("gender")
    is_peak = Variable("is_peak")

    # ASC_car = 0)
    asc_pt = Beta("asc_pt",   0, None, None, 0)
    asc_bike = Beta("asc_bike", 0, None, None, 0)

    # travel-time coefficient
    b_time_pt = Beta("b_time_pt", 0, None, None, 0)
    b_time_bike = Beta("b_time_bike", 0, None, None, 0)

    # Alternative-specific socioeconomic (car = 0)
    b_income_pt = Beta("b_income_pt",   0, None, None, 0)
    b_income_bike = Beta("b_income_bike", 0, None, None, 0)

    # b_license_pt = Beta("b_license_pt",   0, None, None, 0)
    # b_license_bike = Beta("b_license_bike", 0, None, None, 0)

    # b_cars_pt = Beta("b_cars_pt",   0, None, None, 0)
    # b_cars_bike = Beta("b_cars_bike", 0, None, None, 0)

    b_age_pt = Beta("b_age_pt",   0, None, None, 0)
    b_age_bike = Beta("b_age_bike", 0, None, None, 0)

    b_gender_pt = Beta("b_gender_pt",   0, None, None, 0)
    b_gender_bike = Beta("b_gender_bike", 0, None, None, 0)

    # Mode-specific LoS variables
    b_dist_bike = Beta("b_dist_bike", 0, None, None, 0)   # bike only

    # Optional peak-hour (Experiment B) — alternative-specific ===
    if include_peak:
        b_peak_pt = Beta("b_peak_pt",   0, None, None, 0)
        b_peak_bike = Beta("b_peak_bike", 0, None, None, 0)
    else:
        b_peak_pt = 0
        b_peak_bike = 0

    # Utility functions:
    
    # V_car: reference — only generic time enters
    v_car = 0

    # V_pt: ASC + generic time + socioeconomic (alt-specific) + transfers
    v_pt = (
        asc_pt
        + b_time_pt * travel_time_min
        + b_income_pt * income_quintile
        # + b_license_pt * has_driving_lic
        # + b_cars_pt * n_cars
        + b_age_pt * age_band
        + b_gender_pt * gender
        + b_peak_pt * is_peak
    )

    # V_bike: ASC + generic time + socioeconomic (alt-specific) + distance
    v_bike = (
        asc_bike
        + b_time_bike * travel_time_min
        + b_income_bike * income_quintile
        # + b_license_bike * has_driving_lic
        # + b_cars_bike * n_cars
        + b_age_bike * age_band
        + b_gender_bike * gender
        + b_dist_bike * distance_km
        + b_peak_bike * is_peak
    )

    # --- Choice model ---
    V  = {1: v_car, 2: v_pt, 3: v_bike}
    av = {1: 1, 2: 1, 3: 1}          # all modes available
    logprob = models.loglogit(V, av, choice)

    # --- Biogeme estimation ---
    biogeme_model = bio.BIOGEME(database, logprob, generate_html=False)
    biogeme_model.modelName = f"mnl_{city_name.lower()}_{tag}"
    
    results = biogeme_model.estimate()

    # --- Extract results ---
    beta_table = results.get_estimated_parameters()
    beta_table = beta_table.reset_index().rename(columns = {"index": "parameter"})
    beta_table["city"] = city_name
    beta_table["spec"] = tag
    
    # --- Fit statistics ---
    n_obs = city_workset.shape[0]
    ll_null = -n_obs * np.log(3)

    general_stats = results.get_general_statistics()

    # Extract LL and n_params — values may be tuples like (value, precision_str)
    ll_star = None
    n_params = None
    for key, val in general_stats.items():
        key_str = str(key).lower()
        # val is often a tuple: (numeric_value, format_string)
        numeric_val = val[0] if isinstance(val, (tuple, list)) else val
        if 'final log likelihood' in key_str:
            ll_star = float(numeric_val)
        if 'number of estimated parameters' in key_str:
            n_params = int(numeric_val)

    # Fallback
    if ll_star is None:
        try:
            ll_star = float(results.data.logLike)
        except Exception:
            ll_star = np.nan
    if n_params is None:
        n_params = len(beta_table)

    rho2 = 1.0 - (ll_star / ll_null)

    fit_stats = {
        "city": city_name, "spec": tag,
        "n_obs": n_obs, "n_params": n_params,
        "LL_null": round(ll_null, 2),
        "LL_star": round(ll_star, 2),
        "rho2": round(rho2, 4),
    }

    # Print Resutlts
    print(f"\n{'='*60}")
    print(f"✓ {city_name} | {tag} | n={n_obs:,} | LL*={ll_star:.2f} | ρ²={rho2:.4f}")
    print(f"{'='*60}")
    print(beta_table.to_string(index=False))

    return biogeme_model, results, beta_table, fit_stats

In [11]:
amst_model_a, amst_results_a, amst_beta_a, amst_fit_a = estimate_city_mnl_v2(
    amsterdam_workset, "Amsterdam", include_peak = False
)


✓ Amsterdam | exp_a_base | n=681 | LL*=-415.09 | ρ²=0.4452
 parameter          Name     Value  Robust std err.  Robust t-stat.  Robust p-value      city       spec
         0        asc_pt -0.405245         1.034387       -0.391773    6.952261e-01 Amsterdam exp_a_base
         1     b_time_pt  0.055995         0.010942        5.117230    3.100552e-07 Amsterdam exp_a_base
         2   b_income_pt -0.248549         0.114526       -2.170250    2.998792e-02 Amsterdam exp_a_base
         3      b_age_pt -0.322968         0.107425       -3.006456    2.643122e-03 Amsterdam exp_a_base
         4   b_gender_pt  0.546020         0.304088        1.795598    7.255846e-02 Amsterdam exp_a_base
         5      asc_bike  4.941268         0.716896        6.892589    5.478507e-12 Amsterdam exp_a_base
         6   b_time_bike  0.053693         0.010916        4.918874    8.704332e-07 Amsterdam exp_a_base
         7 b_income_bike  0.135610         0.081165        1.670790    9.476312e-02 Amsterdam exp_a_

In [12]:
amst_model_b, amst_results_b, amst_beta_b, amst_fit_b = estimate_city_mnl_v2(
    amsterdam_workset, "Amsterdam", include_peak = True
)


✓ Amsterdam | exp_b_peak | n=681 | LL*=-413.59 | ρ²=0.4472
 parameter          Name     Value  Robust std err.  Robust t-stat.  Robust p-value      city       spec
         0        asc_pt -0.150696         1.038998       -0.145040    8.846797e-01 Amsterdam exp_b_peak
         1     b_time_pt  0.056596         0.011123        5.088077    3.617125e-07 Amsterdam exp_b_peak
         2   b_income_pt -0.216330         0.117657       -1.838652    6.596637e-02 Amsterdam exp_b_peak
         3      b_age_pt -0.350205         0.110312       -3.174662    1.500112e-03 Amsterdam exp_b_peak
         4   b_gender_pt  0.584940         0.309937        1.887286    5.912181e-02 Amsterdam exp_b_peak
         5     b_peak_pt -0.478610         0.309536       -1.546219    1.220516e-01 Amsterdam exp_b_peak
         6      asc_bike  4.950392         0.722791        6.848993    7.437162e-12 Amsterdam exp_b_peak
         7   b_time_bike  0.053852         0.011096        4.853143    1.215199e-06 Amsterdam exp_b_

In [13]:
lnd_model_a, lnd_results_a, lnd_beta_a, lnd_fit_a = estimate_city_mnl_v2(
    london_workset, "London", include_peak = False
)


✓ London | exp_a_base | n=3,315 | LL*=-2573.10 | ρ²=0.2935
 parameter          Name     Value  Robust std err.  Robust t-stat.  Robust p-value   city       spec
         0        asc_pt  1.288782         0.294127        4.381714    1.177494e-05 London exp_a_base
         1     b_time_pt  0.037669         0.002991       12.595296    0.000000e+00 London exp_a_base
         2   b_income_pt  0.053578         0.027417        1.954183    5.067953e-02 London exp_a_base
         3      b_age_pt -0.534003         0.033820      -15.789380    0.000000e+00 London exp_a_base
         4   b_gender_pt  0.744926         0.083001        8.974858    0.000000e+00 London exp_a_base
         5      asc_bike -0.182450         0.516609       -0.353168    7.239622e-01 London exp_a_base
         6   b_time_bike  0.022193         0.004789        4.634050    3.585810e-06 London exp_a_base
         7 b_income_bike  0.342458         0.051833        6.606913    3.924150e-11 London exp_a_base
         8    b_age_bi

In [14]:
lnd_model_b, lnd_results_b, lnd_beta_b, lnd_fit_b = estimate_city_mnl_v2(
    london_workset, "London", include_peak = True
)


✓ London | exp_b_peak | n=3,315 | LL*=-2571.61 | ρ²=0.2939
 parameter          Name     Value  Robust std err.  Robust t-stat.  Robust p-value   city       spec
         0        asc_pt  1.225103         0.299407        4.091766    4.281003e-05 London exp_b_peak
         1     b_time_pt  0.037492         0.002994       12.521058    0.000000e+00 London exp_b_peak
         2   b_income_pt  0.050131         0.027625        1.814664    6.957562e-02 London exp_b_peak
         3      b_age_pt -0.530441         0.033832      -15.678693    0.000000e+00 London exp_b_peak
         4   b_gender_pt  0.740160         0.083118        8.904944    0.000000e+00 London exp_b_peak
         5     b_peak_pt  0.088112         0.093277        0.944630    3.448477e-01 London exp_b_peak
         6      asc_bike -0.370293         0.545459       -0.678866    4.972229e-01 London exp_b_peak
         7   b_time_bike  0.021541         0.004800        4.487812    7.195828e-06 London exp_b_peak
         8 b_income_bi

We can see the improvment in the $R^2$ score for both experiments per city.

Generally, Amsterdam $R^2$ = 0.45 > London $R^2$ = 0.29 indicating that Amsterdam's model explains more variance. We can assume that this is because Amsterdam has a dominant mode (bike = 60%), making it easier to predict especially for short trips. London's split is more balanced (PT 53%, car 39%, bike 8%), so predictions are harder.
Amsterdam B barely improves over A ($R^2$ goes from 0.4452 to 0.4472). London B also barely improves over A (0.2935  to 0.0.2939).

### 4.5 Experiment Selection 
Experiment A is nested within Experiment B (B adds two parameters: `b_peak_pt` and `b_peak_bike`). We use the likelihood-ratio (LR) test to determine whether the additional peak-hour parameters provide a statistically significant improvement in model fit (Ortúzar & Willumsen, 2011, §8.6):

$$LR = -2 \left[ \mathcal{LL}_A - \mathcal{LL}_B \right] \sim \chi^2(\Delta k)$$

where $\Delta k = 2$ (the number of additional parameters). At $\alpha = 0.05$, the critical value is $\chi^2_{0.05}(2) = 5.991$. If $LR > 5.991$, we reject $H_0$ (the restricted model is sufficient) and select Experiment B.If the score is lower, we stick with Experiment A because it's simpler and the extra data didn't add statistical value.

In [15]:
def lr_test(fit_a, fit_b, alpha=0.05):
    """
    Likelihood-ratio test for nested models.
    H0: restricted model (A) is sufficient.
    H1: unrestricted model (B) fits significantly better.
    """
    lr_stat = -2 * (fit_a["LL_star"] - fit_b["LL_star"])
    delta_k = fit_b["n_params"] - fit_a["n_params"]
    p_value = chi2.sf(lr_stat, df=delta_k)
    reject  = p_value < alpha

    print(f" LR statistic : {lr_stat:.4f}")
    print(f" delta_k (df) : {delta_k}")
    print(f" p-value : {p_value:.6f}")
    print(f" Critical chi^2 : {chi2.ppf(1 - alpha, delta_k):.3f}")
    print(f" Decision : {'Reject H0 → select Exp B' if reject else 'Fail to reject H0, so we select Exp A'}")
    return reject, lr_stat, p_value


print("\nAMSTERDAM: LR Test (Exp A vs Exp B)")
amst_select_b, _, _ = lr_test(amst_fit_a, amst_fit_b)

print("\nLONDON: LR Test (Exp A vs Exp B)")
lnd_select_b, _, _ = lr_test(lnd_fit_a, lnd_fit_b)


AMSTERDAM: LR Test (Exp A vs Exp B)
 LR statistic : 3.0000
 delta_k (df) : 2
 p-value : 0.223130
 Critical chi^2 : 5.991
 Decision : Fail to reject H0, so we select Exp A

LONDON: LR Test (Exp A vs Exp B)
 LR statistic : 2.9800
 delta_k (df) : 2
 p-value : 0.225373
 Critical chi^2 : 5.991
 Decision : Fail to reject H0, so we select Exp A


In both of our experiments we fail reject the null hypothesis that adding the b_peak to the model is improving the model and is statistically significant. Generally, the "peak hour" effect isn't strong enough to justify the extra complexity. 

In both experiments the p-value is much higher than 0.05 so we can suggest that any change in mode choice is probably random instead of being affected by the peak hour. In Amsterdam, commuters' choice of transport (bike vs. car/PT) doesn't change significantly just because it's rush hour. Even though London has a much larger sample size, the improvement was still very small.

In [16]:
# --- Store final selections ---
amst_final_tag   = "exp_b_peak" if amst_select_b else "exp_a_base"
amst_final_beta  = amst_beta_b  if amst_select_b else amst_beta_a
amst_final_fit   = amst_fit_b   if amst_select_b else amst_fit_a
amst_final_res   = amst_results_b if amst_select_b else amst_results_a

lnd_final_tag    = "exp_b_peak" if lnd_select_b else "exp_a_base"
lnd_final_beta   = lnd_beta_b   if lnd_select_b else lnd_beta_a
lnd_final_fit    = lnd_fit_b    if lnd_select_b else lnd_fit_a
lnd_final_res    = lnd_results_b if lnd_select_b else lnd_results_a

print(f"\n✓ Amsterdam final model: {amst_final_tag}")
print(f"✓ London final model:    {lnd_final_tag}")


✓ Amsterdam final model: exp_a_base
✓ London final model:    exp_a_base


## 4.6 Comparative Reporting

In this step we are going to consolidate the estimation results into two side-by-side tables for cross-city comparison: 
1. A coefficient table showing parameter estimates, robust standard errors, t-statistics, and p-values for each city's final model;
2. A goodness-of-fit table reporting the number of observations, log-likelihood at convergence, McFadden's $\rho^2$, and the number of estimated parameters.


In [17]:
def format_coeff_table(beta_df, city_label):
    """Format one city's beta table for merging."""
    cols = ["Name", "Value", "Robust std err.", "Robust t-stat.", "Robust p-value"]
    out = beta_df[cols].copy()
    out = out.rename(columns={
        "Value":         f"{city_label}_coeff",
        "Robust std err.":  f"{city_label}_se",
        "Robust t-stat.":   f"{city_label}_t",
        "Robust p-value":  f"{city_label}_p",
    })
    # Round for readability
    for c in out.columns[1:]:
        out[c] = out[c].round(4)
    return out


amst_fmt = format_coeff_table(amst_final_beta, "AMS")
lnd_fmt  = format_coeff_table(lnd_final_beta, "LDN")

In [18]:
# Merge on parameter name (outer join to keep all params from both)
comparison = amst_fmt.merge(lnd_fmt, on = "Name", how = "outer").set_index("Name")

# Reorder columns for readability: AMS coeff, SE, t, p | LDN coeff, SE, t, p
col_order = [
    "AMS_coeff", "AMS_se", "AMS_t", "AMS_p",
    "LDN_coeff", "LDN_se", "LDN_t", "LDN_p",
]
comparison = comparison[[c for c in col_order if c in comparison.columns]]

In [19]:
print("=" * 90)
print("SIDE-BY-SIDE COEFFICIENT TABLE (Final Models)")
print("=" * 90)
print(comparison.to_string())


# --- Goodness-of-fit summary ---
fit_summary = pd.DataFrame([amst_final_fit, lnd_final_fit])
fit_summary = fit_summary.set_index("city")[["spec", "n_obs", "n_params", "LL_null", "LL_star", "rho2"]]

print(f"\n{'=' * 90}")
print("GOODNESS-OF-FIT SUMMARY")
print("=" * 90)
print(fit_summary.to_string())

SIDE-BY-SIDE COEFFICIENT TABLE (Final Models)
               AMS_coeff  AMS_se    AMS_t   AMS_p  LDN_coeff  LDN_se    LDN_t   LDN_p
Name                                                                                 
asc_bike          4.9413  0.7169   6.8926  0.0000    -0.1825  0.5166  -0.3532  0.7240
asc_pt           -0.4052  1.0344  -0.3918  0.6952     1.2888  0.2941   4.3817  0.0000
b_age_bike       -0.3226  0.0720  -4.4836  0.0000    -0.2590  0.0491  -5.2743  0.0000
b_age_pt         -0.3230  0.1074  -3.0065  0.0026    -0.5340  0.0338 -15.7894  0.0000
b_dist_bike      -0.3892  0.0383 -10.1525  0.0000    -0.1477  0.0121 -12.2005  0.0000
b_gender_bike    -0.4910  0.2219  -2.2130  0.0269    -0.1456  0.1399  -1.0410  0.2979
b_gender_pt       0.5460  0.3041   1.7956  0.0726     0.7449  0.0830   8.9749  0.0000
b_income_bike     0.1356  0.0812   1.6708  0.0948     0.3425  0.0518   6.6069  0.0000
b_income_pt      -0.2485  0.1145  -2.1702  0.0300     0.0536  0.0274   1.9542  0.0507
b_time_b

Generally, Amsterdam model explains nearly 45% of the choice behavior (rho2 = 0.4452) while Londons model slightly below 30% (rho2 = 0.2935). It is lower than Amsterdam’s, likely because London is more complex and has more "random" travel factors.

In Amsterdam we have very high positive number for bike asc_bike = 4.94 meaning that even if time and cost were equal, Amsterdam commuters have a "default" preference for biking. In London, it is slightly negative (-0.18), meaning there is a slight psychological barrier to biking compared to driving.

As expected, distance (b_dist_bike) is negative in both cities. However, it is much more negative in Amsterdam (-0.38) than London (-0.14) suggesting that  Amsterdam commuters are more sensitive to distance; they will bike short trips but are very quick to switch to another mode as the distance grows.

Regarding PT, London has a strong positive preference for PT (1.28). People "default" to the PT while Amsterdam has a negative value (-0.40). However, the p-value (0.69) of asc_pt says this isn't statistically significant meaning that Amsterdam commuters don't necessarily "hate" PT, they just prefer biking.

## 4.7 Interpretation for Hypotheses

This section explains how the study’s results relate to its main goals. It is important to note that these results show links between factors, not that one thing definitely causes another.
To compare cities, we look at the size of the numbers (coefficients) and check if their estimated ranges (confidence intervals) overlap.

1. H1 — How people feel about travel time?
   1. We are looking at how much people dislike long trips. The Goal is to  compare travel time data between cities. If London has a more negative number, it means commuters there are more negatively affected by travel time. This is likely because London transport network is very large and traffic is unpredictable.
2. H2 — How car ownership affects travel choices?
    1. We are looking at whether owning a car stops people from using public transport or bikes.We want to see if "having a car" matters less in London or Amsterdam. We expect that  Amsterdam will have a "weaker" effect, because in Amsterdam, cycling is very common so people choose to bike even if they have a car parked at home.


In [20]:
def extract_param(beta_df, param_name):
    """Extract estimate and SE for a single parameter."""
    row = beta_df.loc[beta_df["Name"] == param_name]
    if row.empty:
        return np.nan, np.nan
    return row["Value"].values[0], row["Robust std err."].values[0]
    
def analyze_hypothesis(param_name, ams_beta, lnd_beta, title):
    print(f"\n{title}")
    
    b_ams, se_ams = extract_param(ams_beta, param_name)
    b_lnd, se_lnd = extract_param(lnd_beta, param_name)

    # Confidence Intervals Calculation
    ci_ams = (b_ams - 1.96 * se_ams, b_ams + 1.96 * se_ams)
    ci_lnd = (b_lnd - 1.96 * se_lnd, b_lnd + 1.96 * se_lnd)

    # 3. Results
    print(f"  Amsterdam  {param_name} = {b_ams:.4f} (SE={se_ams:.4f}) 95% CI: [{ci_ams[0]:.4f}, {ci_ams[1]:.4f}]")
    print(f"  London     {param_name} = {b_lnd:.4f} (SE={se_lnd:.4f}) 95% CI: [{ci_lnd[0]:.4f}, {ci_lnd[1]:.4f}]")

    # 4. Overlap Logic
    overlap = ci_ams[1] >= ci_lnd[0] and ci_lnd[1] >= ci_ams[0]
    print(f"  CIs overlap: {overlap}")
    
    if not overlap:
        print("  → Coefficients are significantly different at 95% level.")
    else:
        print("  → CIs overlap — formal Wald test needed")

In [22]:
# --- EXECUTION ---

# H1 Analysis
analyze_hypothesis("b_time_pt", amst_final_beta, lnd_final_beta, "H1 — TRAVEL TIME SENSITIVITY (PT)")
analyze_hypothesis("b_time_bike", amst_final_beta, lnd_final_beta, "H1 — TRAVEL TIME SENSITIVITY (BIKE)")

# H2 Analysis
print(f"\nH2 — AGE & INCOME EFFECT\n")
h2_params = ["b_age_pt", "b_age_bike", "b_income_pt", "b_income_bike"]

for param in h2_params:
    val_a, se_a = extract_param(amst_final_beta, param)
    val_l, se_l = extract_param(lnd_final_beta, param)
    print(f"  {param:15s} | AMS: {val_a:+.4f} (SE={se_a:.4f}) | LDN: {val_l:+.4f} (SE={se_l:.4f})")


H1 — TRAVEL TIME SENSITIVITY (PT)
  Amsterdam  b_time_pt = 0.0560 (SE=0.0109) 95% CI: [0.0345, 0.0774]
  London     b_time_pt = 0.0377 (SE=0.0030) 95% CI: [0.0318, 0.0435]
  CIs overlap: True
  → CIs overlap — formal Wald test needed

H1 — TRAVEL TIME SENSITIVITY (BIKE)
  Amsterdam  b_time_bike = 0.0537 (SE=0.0109) 95% CI: [0.0323, 0.0751]
  London     b_time_bike = 0.0222 (SE=0.0048) 95% CI: [0.0128, 0.0316]
  CIs overlap: False
  → Coefficients are significantly different at 95% level.

H2 — AGE & INCOME EFFECT

  b_age_pt        | AMS: -0.3230 (SE=0.1074) | LDN: -0.5340 (SE=0.0338)
  b_age_bike      | AMS: -0.3226 (SE=0.0720) | LDN: -0.2590 (SE=0.0491)
  b_income_pt     | AMS: -0.2485 (SE=0.1145) | LDN: +0.0536 (SE=0.0274)
  b_income_bike   | AMS: +0.1356 (SE=0.0812) | LDN: +0.3425 (SE=0.0518)


In [25]:
amst_final_beta

,parameter,Name,Value,Robust std err.,Robust t-stat.,Robust p-value,city,spec
0,0,asc_pt,-0.405245,1.034387,-0.391773,6.952261e-01,Amsterdam,exp_a_base
1,1,b_time_pt,0.055995,0.010942,5.117230,3.100552e-07,Amsterdam,exp_a_base
2,2,b_income_pt,-0.248549,0.114526,-2.170250,2.998792e-02,Amsterdam,exp_a_base
3,3,b_age_pt,-0.322968,0.107425,-3.006456,2.643122e-03,Amsterdam,exp_a_base
4,4,b_gender_pt,0.546020,0.304088,1.795598,7.255846e-02,Amsterdam,exp_a_base
5,5,asc_bike,4.941268,0.716896,6.892589,5.478507e-12,Amsterdam,exp_a_base
6,6,b_time_bike,0.053693,0.010916,4.918874,8.704332e-07,Amsterdam,exp_a_base
7,7,b_income_bike,0.135610,0.081165,1.670790,9.476312e-02,Amsterdam,exp_a_base
8,8,b_age_bike,-0.322620,0.071956,-4.483582,7.340024e-06,Amsterdam,exp_a_base
9,9,b_gender_bike,-0.490989,0.221869,-2.212971,2.689965e-02,Amsterdam,exp_a_base


In [26]:
# --- Export ---
amst_final_beta.to_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\amst_final_beta.csv", index=False)
print(f"\n Amsterdam model saved to data\processed")


 Amsterdam model saved to data\processed


In [29]:
# --- Export ---
lnd_final_beta.to_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\lnd_final_beta.csv", index=False)
print(f"\n London model saved to data\processed")


 London model saved to data\processed


## References

Ben-Akiva, M. E., & Lerman, S. R. (1985). *Discrete choice analysis: Theory and application to travel demand*. MIT Press.

Bierlaire, M. (2020). A short introduction to PandasBiogeme (Technical Report TRANSP-OR 200605). Transport and Mobility Laboratory, EPFL. https://transp-or.epfl.ch/documents/technicalReports/Bier20.pdf

Harms, L., Bertolini, L., & te Brömmelstroet, M. (2014). Training and explaining travel behaviour changes: An experiment with a peer-to-peer smartphone app and travel diary in the Netherlands. *Journal of Transport & Health*, *1*(2), 134–142. https://doi.org/10.1016/j.jth.2014.03.003

Hensher, D. A., Rose, J. M., & Greene, W. H. (2015). *Applied choice analysis* (2nd ed.). Cambridge University Press.

McFadden, D. (1974). Conditional logit analysis of qualitative choice behavior. In P. Zarembka (Ed.), *Frontiers in econometrics* (pp. 105–142). Academic Press.

Ortúzar, J. de D., & Willumsen, L. G. (2011). *Modelling transport* (4th ed.). John Wiley & Sons.

Train, K. E. (2009). *Discrete choice methods with simulation* (2nd ed.). Cambridge University Press.

Wardman, M. (2004). Public transport values of time. *Transport Policy*, *11*(4), 363–377. https://doi.org/10.1016/j.tranpol.2004.05.001

Wardman, M., Chintakayala, V. P., & de Jong, G. (2016). Values of travel time in Europe: Review and meta-analysis. *Transportation Research Part A: Policy and Practice*, *94*, 93–111. https://doi.org/10.1016/j.tra.2016.08.019

Wickham, H. (2014). Tidy data. *Journal of Statistical Software*, *59*(10), 1–23. https://doi.org/10.18637/jss.v059.i10